# Echo-TOF WIFF2 Analysis Pipeline
**echo_tof + echo_tof_ext 실제 로직 기반 워크플로우**

| Part | 내용 | 핵심 모듈 |
|------|------|-----------|
| 0 | 환경 설정 | pip install |
| 1 | 데이터 로드 & 가시화 | matplotlib |
| 2 | 피크 검출 | di_spectrum.pick_peaks |
| 3 | 동위원소 패턴 계산 | IsotopicDistributionCalculator |
| 4 | 동위원소 검증 | IsotopeVerifier |
| 5 | 분자식 열거 & 필터링 | FormulaFilter |
| 6 | 패턴 매칭 & 순위 | FormulaFinderPipeline |
| 7 | 반응 예측 & 피크 분류 | reaction_predictor, PeakClassifier |
| 8 | 종합 리포트 | summary |

## Part 0. 환경 설정

In [ ]:
!pip install -q numpy scipy matplotlib seaborn pandas

In [ ]:
# echo_tof 패키지 로드
!git clone -q https://github.com/chsong-hash/echo-tof-colab.git /content/etof 2>/dev/null || true
import sys, os
for p in ['/content/etof/src', '/content/echo-tof-colab/src', '.', '..']:
    if os.path.isdir(p): sys.path.insert(0, p)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import find_peaks
from scipy.ndimage import uniform_filter1d, minimum_filter1d

plt.rcParams['figure.figsize'] = (13, 4)
plt.rcParams['font.size'] = 10
sns.set_style("whitegrid")

# echo_tof core
try:
    from echo_tof.elements import PeriodicTable, parse_formula, get_monoisotopic_mass, calculate_rdb
    from echo_tof.molecule import Molecule
    from echo_tof.isotope_calc import IsotopicDistributionCalculator
    from echo_tof.pattern import MoleculePattern
    from echo_tof.filters import FormulaFilter
    from echo_tof.pipeline import FormulaFinderPipeline, CompositionResult
    from echo_tof.calculations import get_mass_error
    HAS_ECHOTOF = True
    print("echo_tof core loaded")
except ImportError as e:
    HAS_ECHOTOF = False
    print(f"echo_tof not available: {e}")

# echo_tof_ext
try:
    from echo_tof_ext.di_spectrum import pick_peaks, group_isotope_clusters, extract_cluster_at_mz
    from echo_tof_ext.isotope_verifier import IsotopeVerifier
    from echo_tof_ext.mz_predict import predict_mz
    from echo_tof_ext.reaction_predictor import predict_product_mw, predict_byproduct_mws
    from echo_tof_ext.neutral_loss_db import NEUTRAL_LOSSES, DELTA_MZ_PATTERNS
    HAS_EXT = True
    print("echo_tof_ext loaded")
except ImportError as e:
    HAS_EXT = False
    print(f"echo_tof_ext not available: {e}")

print("\nSetup complete!")

## Part 1. 데이터 로드 & 스펙트럼 가시화
WIFF2를 mzML로 변환 후 사용 권장. 데모용 Caffeine (C8H10N4O2) 스펙트럼을 시뮬레이션합니다.

In [ ]:
# Caffeine 시뮬레이션 스펙트럼 (동위원소 + 프래그먼트 + adduct 포함)
np.random.seed(42)
mz = np.linspace(50, 500, 5000)
intensity = np.zeros_like(mz)

PEAKS = [
    (195.0877, 10000, 0.015, "[M+H]+"),
    (196.0911,   700, 0.015, "M+1 (13C)"),
    (197.0944,    30, 0.015, "M+2"),
    (217.0697,  1500, 0.015, "[M+Na]+"),
    (138.0662,  4500, 0.015, "frag C5H7N3O+"),
    (110.0713,  2000, 0.015, "frag C4H7N3+"),
    (82.0531,    800, 0.015, "frag C3H5N3+"),
    (42.0338,    300, 0.015, "frag C2H3N+"),
]
for center, height, width, _ in PEAKS:
    intensity += height * np.exp(-((mz - center)**2) / (2 * width**2))
intensity += np.abs(np.random.normal(0, 30, len(mz)))

print(f"Spectrum: {len(mz)} points, m/z [{mz[0]:.0f}-{mz[-1]:.0f}]")
for c, h, _, label in PEAKS:
    print(f"  {c:.4f}  {h:>6.0f}  {label}")

In [ ]:
# 스펙트럼 가시화
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

ax = axes[0]
ax.plot(mz, intensity, color='#0078D4', linewidth=0.5)
ax.set_xlabel('m/z'); ax.set_ylabel('Intensity'); ax.set_title('Full Spectrum')
for c, h, _, lbl in PEAKS:
    if h > 500:
        idx = np.argmin(np.abs(mz - c))
        ax.annotate(f'{c:.3f}', xy=(c, intensity[idx]), xytext=(0,8), textcoords='offset points', ha='center', fontsize=6, color='red')

ax = axes[1]
m1 = (mz >= 194) & (mz <= 199)
ax.plot(mz[m1], intensity[m1], color='#E85D04', linewidth=1)
ax.set_xlabel('m/z'); ax.set_title('[M+H]+ Isotope Cluster')

ax = axes[2]
m2 = (mz >= 30) & (mz <= 160)
ax.plot(mz[m2], intensity[m2], color='#2D9F2D', linewidth=0.8)
ax.set_xlabel('m/z'); ax.set_title('Fragment Region')

plt.tight_layout(); plt.show()

## Part 2. 피크 검출
di_spectrum.pick_peaks() - MAD 기반 노이즈 추정, S/N 필터링, 로컬 최댓값 검출

In [ ]:
# 피크 검출
def standalone_pick_peaks(mz, intensity, noise_factor=3.0, min_intensity_pct=0.1, local_window=5):
    sorted_int = np.sort(intensity)
    noise = np.median(sorted_int[:len(sorted_int)//2])
    if noise <= 0: noise = np.median(np.abs(np.diff(intensity))) * 1.4826
    threshold = max(noise * noise_factor, np.max(intensity) * min_intensity_pct / 100)
    peaks_idx, _ = find_peaks(intensity, height=threshold, distance=local_window, prominence=noise*2)
    results = []
    for idx in peaks_idx:
        left = max(0, idx - local_window)
        right = min(len(mz)-1, idx + local_window)
        area = np.trapz(intensity[left:right+1], mz[left:right+1])
        sn = intensity[idx] / noise if noise > 0 else 0
        results.append({'mz': mz[idx], 'intensity': intensity[idx], 'area': area, 'sn': sn, 'index': idx})
    return results

if HAS_EXT:
    detected = pick_peaks(mz, intensity, noise_factor=3.0, min_intensity_pct=0.1)
else:
    detected = standalone_pick_peaks(mz, intensity)

peaks_df = pd.DataFrame(detected)
print(f"Detected {len(peaks_df)} peaks")
peaks_df[['mz','intensity','area','sn']].round(2)

In [ ]:
# 피크 가시화
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(mz, intensity, color='#888', linewidth=0.5, alpha=0.5)
for _, pk in peaks_df.iterrows():
    ax.plot(pk['mz'], pk['intensity'], 'rv', markersize=6)
    ax.annotate(f"{pk['mz']:.3f}\nS/N={pk['sn']:.0f}", xy=(pk['mz'], pk['intensity']),
               xytext=(0,12), textcoords='offset points', ha='center', fontsize=6, color='red')
ax.set_xlabel('m/z'); ax.set_ylabel('Intensity')
ax.set_title(f'Peak Detection: {len(peaks_df)} peaks')
plt.tight_layout(); plt.show()

In [ ]:
# 동위원소 클러스터링
ISOTOPE_SPACING = 1.003355
if HAS_EXT:
    clusters = group_isotope_clusters(detected, charge=1, mz_tolerance=0.02)
else:
    clusters = []
    used = set()
    for pk in sorted(detected, key=lambda p: -p['intensity']):
        if pk['mz'] in used: continue
        cl_peaks = [pk]; used.add(pk['mz'])
        for other in detected:
            if other['mz'] in used: continue
            for n in range(1, 6):
                if abs(other['mz'] - pk['mz'] - n * ISOTOPE_SPACING) < 0.02:
                    cl_peaks.append(other); used.add(other['mz']); break
        clusters.append({'mono_mz': pk['mz'], 'peaks': cl_peaks})

print(f"Clusters: {len(clusters)}")
for cl in clusters:
    print(f"  mono={cl['mono_mz']:.4f}, {len(cl['peaks'])} peaks")

## Part 3. 동위원소 패턴 계산
IsotopicDistributionCalculator - 이진 지수화 Convolution으로 이론적 동위원소 분포를 계산합니다.

In [ ]:
FORMULA = "C8H10N4O2"  # Caffeine
CHARGE = 1

if HAS_ECHOTOF:
    idc = IsotopicDistributionCalculator(ppm_tolerance=True, tolerance=50.0)
    mol = Molecule(FORMULA, charge=CHARGE)
    mp = MoleculePattern(charge=CHARGE)
    mp.calculate_pattern(mol, idc)
    theor_mz = list(mp.pattern_mz)
    theor_int = list(mp.pattern_rel_intensities)
    print(f"Molecule: {mol.composition}, MW={mol.monoisotopic_mass:.6f}")
    print(f"[M+H]+ m/z = {mol.mono_mass_to_charge:.6f}, RDB={mol.rdb:.1f}")
else:
    theor_mz = [195.0877, 196.0911, 197.0944]
    theor_int = [100.0, 9.2, 0.5]
    print(f"Standalone: {FORMULA}")

print("\nIsotope pattern:")
for m, i in zip(theor_mz, theor_int):
    print(f"  m/z {m:.4f}  {i:6.1f}%  {'#' * int(i/2)}")

In [ ]:
# 이론 vs 실측 오버레이
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

ax = axes[0]
ax.bar(theor_mz, theor_int, width=0.12, color='#2196F3', alpha=0.8, label='Theoretical')
for m, i in zip(theor_mz, theor_int):
    ax.annotate(f'{m:.4f}\n({i:.1f}%)', xy=(m,i), xytext=(0,5), textcoords='offset points', ha='center', fontsize=7)
ax.set_xlabel('m/z'); ax.set_ylabel('Rel. Int (%)'); ax.set_title(f'{FORMULA} Theoretical'); ax.legend()

ax = axes[1]
mask = (mz >= theor_mz[0]-1) & (mz <= theor_mz[-1]+1)
exp_norm = intensity[mask] / np.max(intensity[mask]) * 100
ax.plot(mz[mask], exp_norm, color='#333', linewidth=1, alpha=0.7, label='Experimental')
ax.bar(theor_mz, theor_int, width=0.08, color='#E85D04', alpha=0.6, label='Theoretical')
ax.set_xlabel('m/z'); ax.set_ylabel('Rel. Int (%)'); ax.set_title('Theory vs Experiment'); ax.legend()

plt.tight_layout(); plt.show()

## Part 4. 동위원소 검증
IsotopeVerifier - 실측 클러스터 추출 후 이론 패턴과 비교하여 Grade 판정

In [ ]:
# 실측 클러스터 추출
target = 195.0877
exp_mz_list, exp_int_list = [], []
for n in range(5):
    expected = target + n * ISOTOPE_SPACING
    m = np.abs(mz - expected) <= 0.05
    if m.any():
        idx = np.argmax(intensity[m])
        local_int = intensity[m][idx]
        if local_int > np.median(intensity) * 3:
            exp_mz_list.append(mz[m][idx])
            exp_int_list.append(local_int)

max_i = max(exp_int_list) if exp_int_list else 1
exp_int_norm = [i/max_i*100 for i in exp_int_list]
print("Extracted cluster:")
for m, i in zip(exp_mz_list, exp_int_norm):
    print(f"  m/z={m:.4f}, {i:.1f}%")

In [ ]:
# 동위원소 검증
if HAS_EXT and HAS_ECHOTOF:
    verifier = IsotopeVerifier(ppm_tolerance=10.0)
    vresult = verifier.verify(FORMULA, CHARGE, exp_mz_list, exp_int_norm)
    grade = vresult['grade']
    print(f"Grade: {grade.upper()}")
    print(f"Cluster Error: {vresult['cluster_error']:.4f}")
    print(f"RMS Error: {vresult['rms_error_ppm']:.2f} ppm")
    print(f"Peaks: {vresult['n_matched_peaks']}/{vresult['n_total_peaks']}")
else:
    errors = [abs(t-e) for t,e in zip(theor_int[:len(exp_int_norm)], exp_int_norm)]
    rms = np.sqrt(np.mean(np.array(errors)**2)) if errors else 0
    mass_err = abs(exp_mz_list[0]-theor_mz[0])/theor_mz[0]*1e6 if exp_mz_list else 0
    grade = 'excellent' if rms<5 and mass_err<3 else 'good' if rms<15 else 'fair'
    print(f"Grade: {grade} (standalone)")
    print(f"Intensity RMS: {rms:.2f}%, Mass err: {mass_err:.2f} ppm")

## Part 5. 분자식 열거 & 필터링
FormulaFilter - 5단계 필터 (DBE, 전자상태, 헤테로비율, 다원소규칙, 원소비규칙)

In [ ]:
TARGET_MZ = 195.0877
if HAS_ECHOTOF:
    ff = FormulaFilter(mass_tolerance_ppm=5.0, int_tolerance=100.0, dbe_from=-0.5, dbe_to=40.0, even_electron=True, odd_electron=False, common_rules=True)
    candidates = ff.get_compositions("", "C20 H30 N10 O10 S3", TARGET_MZ, CHARGE)
    print(f"Found {len(candidates)} candidates for m/z {TARGET_MZ}")
    rows = []
    for mol in candidates:
        err = get_mass_error(mol.monoisotopic_mass, TARGET_MZ, in_ppm=True)
        rows.append({'Formula': mol.composition, 'MW': f"{mol.monoisotopic_mass:.6f}", 'RDB': f"{mol.rdb:.1f}", 'ppm': f"{err:.1f}", 'Even': mol.is_even_electron})
    pd.DataFrame(rows)
else:
    print("FormulaFilter not available. Known: C8H10N4O2 -> 195.0877")

## Part 6. 패턴 매칭 & 순위
FormulaFinderPipeline - 4가지 오차(ClusterErr, RMS, RMS+Int, MassCluster)로 순위 산출

In [ ]:
if HAS_ECHOTOF and len(exp_mz_list) >= 2:
    pipeline = FormulaFinderPipeline()
    pipeline.init_ms(mass_tol_ppm=5.0, int_tol=100.0, c_to_hetero=0.0, min_comp="", max_comp="C20 H30 N10 O10 S3", dbe_from=-0.5, dbe_to=40.0, even_electron=True, odd_electron=False, common_rules=True)
    pipeline.set_pattern(exp_mz_list, exp_int_norm, [True]*len(exp_mz_list), [1.0]*len(exp_mz_list), CHARGE)
    count = pipeline.propose_elemental_compositions()
    print(f"Proposed {count} compositions")
    if pipeline.results:
        rows = [{'Rank':r.overall_order, 'Formula':r.composition, 'ppm':f"{r.error_ppm:.1f}", 'ClusterErr':f"{r.intensity_cluster_error:.3f}", 'RMS':f"{r.rms_error:.2f}"} for r in sorted(pipeline.results, key=lambda x: x.overall_order)]
        display(pd.DataFrame(rows).head(10))
else:
    print("Requires echo_tof core + isotope data")

In [ ]:
# Top 3 후보 패턴 비교
if HAS_ECHOTOF and hasattr(pipeline, 'results') and pipeline.results:
    top3 = sorted(pipeline.results, key=lambda x: x.overall_order)[:3]
    fig, axes = plt.subplots(1, len(top3), figsize=(5*len(top3), 4))
    if len(top3)==1: axes=[axes]
    for ax, r in zip(axes, top3):
        m = Molecule(r.composition, charge=CHARGE)
        p = MoleculePattern(charge=CHARGE)
        p.calculate_pattern(m, idc)
        ax.bar(p.pattern_mz, p.pattern_rel_intensities, width=0.08, color='#2196F3', alpha=0.6, label='Theory')
        ax.scatter(exp_mz_list, exp_int_norm, color='red', s=40, zorder=5, label='Exp')
        ax.set_title(f"{r.composition}\nRank={r.overall_order}, ppm={r.error_ppm:.1f}", fontsize=9)
        ax.set_xlabel('m/z'); ax.legend(fontsize=7)
    plt.tight_layout(); plt.show()

## Part 7. 반응 예측 & 피크 분류

In [ ]:
SM_MW = 197.084
REAGENT_MW = 136.052
PROTON = 1.00728

if HAS_EXT:
    product_mw = predict_product_mw(SM_MW, 'amide_coupling', REAGENT_MW)
    byproducts = predict_byproduct_mws(SM_MW, 'amide_coupling', REAGENT_MW)
    adducts = predict_mz(product_mw, mode='positive')
    print(f"Product MW: {product_mw:.3f}")
    print("Adducts:")
    for a in adducts: print(f"  {a['adduct']:15s} m/z={a['mz']:.4f}")
    print(f"Byproducts: {len(byproducts)}")
    for bp in byproducts: print(f"  {bp['name']}: MW={bp['mw']:.1f}")
else:
    product_mw = SM_MW + REAGENT_MW - 18.011
    print(f"Product MW ~ {product_mw:.3f}, [M+H]+ = {product_mw+PROTON:.4f}")

In [ ]:
# 피크 분류 가시화
COLOR_MAP = {'Known':'#2E8B57', 'Inferred':'#FF8C00', 'Unknown':'#999'}
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(mz, intensity, color='#ccc', linewidth=0.5)

known_targets = {'SM [M+H]+': SM_MW+PROTON, 'Product [M+H]+': (product_mw or 0)+PROTON}
for _, pk in peaks_df.iterrows():
    cat = 'Unknown'; lbl = ''
    for name, tmz in known_targets.items():
        if abs(pk['mz']-tmz) < 0.1: cat='Known'; lbl=name; break
    if cat=='Unknown' and HAS_EXT:
        for nl_name, nl_mass, _ in NEUTRAL_LOSSES[:20]:
            for tn, tm in known_targets.items():
                if abs(pk['mz']-(tm-nl_mass)) < 0.1: cat='Inferred'; lbl=f'{nl_name} from {tn}'; break
    c = COLOR_MAP[cat]
    ax.plot([pk['mz']]*2, [0, pk['intensity']], color=c, linewidth=2, alpha=0.7)
    if pk['intensity'] > np.max(intensity)*0.05:
        ax.annotate(f"{pk['mz']:.3f}\n{lbl or cat}", xy=(pk['mz'],pk['intensity']), xytext=(0,10), textcoords='offset points', ha='center', fontsize=6, color=c, fontweight='bold')

from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=c,label=n) for n,c in COLOR_MAP.items()])
ax.set_xlabel('m/z'); ax.set_ylabel('Intensity'); ax.set_title('Peak Classification')
plt.tight_layout(); plt.show()

## Part 8. 종합 리포트

In [ ]:
print("="*60)
print("  Echo-TOF Analysis Report")
print("="*60)
print(f"\n[Spectrum] {len(mz)} pts, {len(peaks_df)} peaks, {len(clusters)} clusters")
print(f"[Target] {FORMULA}, [M+{CHARGE}H]{CHARGE}+ = {theor_mz[0]:.4f}")
print(f"[Verification] Grade={grade}")
if HAS_ECHOTOF and hasattr(pipeline,'results') and pipeline.results:
    best = sorted(pipeline.results, key=lambda x: x.overall_order)[0]
    print(f"[Formula Search] {len(pipeline.results)} candidates, Best: {best.composition} (ppm={best.error_ppm:.1f})")
print(f"[Reaction] Product MW={product_mw:.3f}" if product_mw else "")
print("\nDone!")

---
**다음 단계**
1. 실제 WIFF2 데이터: msconvert로 mzML 변환 후 사용
2. echo_tof 패키지: src/ 폴더에 복사 후 import
3. 96웰 배치 분석: EchoPipeline으로 플레이트 전체 자동 분석
4. 반응 조건 최적화: 수율 히트맵으로 최적 조건 탐색